### RAG Pipelines - Data Ingestion to Vector DB Pipeline

In [5]:
import os
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [ ]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: home feb 26.pdf
  ✓ Loaded 1 pages

Processing: Extracting Data using SQL.pdf
  ✓ Loaded 18 pages

Total documents loaded: 19


In [7]:
all_pdf_documents

[Document(metadata={'producer': 'Skia/PDF m145', 'creator': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36', 'creationdate': '2026-02-26T16:15:01+00:00', 'title': 'UHBVN', 'moddate': '2026-02-26T16:15:01+00:00', 'source': '../data/pdf/home feb 26.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'home feb 26.pdf', 'file_type': 'pdf'}, page_content='BILL RECEIPT(PAYMENT MADE THROUGH CASH COLLECTION COUNTER)\nREFERENCE NO125809763 SUB DIVISION City panipat REFERENCE DATE26/02/2026 21:47:38\nParticulars of Bills Original\nS NoTransaction No Account No Consumer Name Sub Division Bill Cycle Payable Amount Rs\n1 287184592 6379250000 ANIL KUMAR City panipat 870\nTotal Amount Paid 870\nParticulars of Payment\nS NoMode Instrument No Instrument Date BankBranch CodeAmount\n1 Credit Card/Debit Card/Net Banking CHMP0JJ1D9G2KC 26/02/2026 HMP 870.00\nTotal Amount Paid (In Figures) 870.00\nTotal Amount Paid (In 

In [ ]:
### text splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """SPLIT DOCUMENTS INTO SMALLER CHUNKS FOR BETTER RAG PERFORMANCE"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators = ["\n\n", '\n', ' ', ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    #show ecxample of a chunk
    if split_docs:
        print(f'\n Example chunk:')
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

chunks = split_documents(all_pdf_documents)

Split 19 documents into 22 chunks

 Ecample chunk:
Content: BILL RECEIPT(PAYMENT MADE THROUGH CASH COLLECTION COUNTER)
REFERENCE NO125809763 SUB DIVISION City panipat REFERENCE DATE26/02/2026 21:47:38
Particulars of Bills Original
S NoTransaction No Account No...
Metadata: {'producer': 'Skia/PDF m145', 'creator': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36', 'creationdate': '2026-02-26T16:15:01+00:00', 'title': 'UHBVN', 'moddate': '2026-02-26T16:15:01+00:00', 'source': '../data/pdf/home feb 26.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'home feb 26.pdf', 'file_type': 'pdf'}


### Embedding and VectorStore DB

In [10]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity 

In [11]:
class EmbeddingManager:
    """Handles document embedding generatiion using SentenceTransformers"""
    
    def __init__(self, model_name: str = "all-MiniLM-l6-v2"):
        """ 
        Initialize the embedding manager
        Args:
            model_name: HuggingFace Model name for sentence embeddings
        """
        self.model_name=model_name
        self.model=None
        self._load_model()
        
    def _load_model(self):
        """Load the SentenceTransformer Model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
            
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
        
    
    def generate_embeddings(self, texts:List[str]) -> np.ndarray:
        """ 
        Generate embeddings for a list of texts
        Args: 
            texts: List of text strings to embed
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model Not Loaded")
        
        print(f"Generating embeddings for {len(texts)} texts ...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
    def get_embedding_dimensions(self) -> int:
        """Get the embedding dimensions of the model."""
        if not self.model:
            raise ValueError("Model not loaded")
        return self.model.get_sentence_embedding_dimension()
    
    
    
### initialise the embedding manager 

embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-l6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9485.21it/s]


Model loaded successfully. Embedding dimension: 384


/var/folders/bm/lz9yd7qn1x7314ld_7qs2djw0000gn/T/ipykernel_1759/2647595836.py:19: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### Vector Store

In [20]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [14]:
chunks

[Document(metadata={'producer': 'Skia/PDF m145', 'creator': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36', 'creationdate': '2026-02-26T16:15:01+00:00', 'title': 'UHBVN', 'moddate': '2026-02-26T16:15:01+00:00', 'source': '../data/pdf/home feb 26.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'home feb 26.pdf', 'file_type': 'pdf'}, page_content='BILL RECEIPT(PAYMENT MADE THROUGH CASH COLLECTION COUNTER)\nREFERENCE NO125809763 SUB DIVISION City panipat REFERENCE DATE26/02/2026 21:47:38\nParticulars of Bills Original\nS NoTransaction No Account No Consumer Name Sub Division Bill Cycle Payable Amount Rs\n1 287184592 6379250000 ANIL KUMAR City panipat 870\nTotal Amount Paid 870\nParticulars of Payment\nS NoMode Instrument No Instrument Date BankBranch CodeAmount\n1 Credit Card/Debit Card/Net Banking CHMP0JJ1D9G2KC 26/02/2026 HMP 870.00\nTotal Amount Paid (In Figures) 870.00\nTotal Amount Paid (In 

In [21]:
### convert the text to embeddings
texts = [doc.page_content for doc in chunks]

## generate the embeddings
embeddings = embedding_manager.generate_embeddings(texts)

## store in the vector database
vectorstore.add_documents(chunks, embeddings)


Generating embeddings for 22 texts ...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.98it/s]

Generated embeddings with shape: (22, 384)
Adding 22 documents to vector store...
Successfully added 22 documents to vector store
Total documents in collection: 22


### Retriever Pipeline From Vector Store

In [27]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

rag_retriever

In [34]:
rag_retriever.retrieve("How much amount is to be paid for the bill?")

Retrieving documents for query: 'How much amount is to be paid for the bill?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts ...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.44it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)


[]